In [1]:
import os
import json
import hmac
import time
import base64
import hashlib
import requests
from dotenv import load_dotenv
from datetime import datetime, timezone
from urllib.parse import urlencode, quote, unquote
from email.utils import format_datetime

load_dotenv()
pudu_api_key = os.getenv('Pd_key')
pudu_api_secret = os.getenv('Pd_secret')
Aurotek_id = os.getenv('Aurotek_id')
BellabotPro = "8BR054C02050014"
Flashbot = "8FF055923050007"

In [ ]:
class PuduAuth:

    @staticmethod
    def utc_date():
        return format_datetime(datetime.now(timezone.utc), usegmt=True)

    @staticmethod
    def md5_base64(content: str) -> str:
        md5_hex = hashlib.md5(content.encode("utf-8")).hexdigest()
        return base64.b64encode(md5_hex.encode("utf-8")).decode("utf-8")

    @staticmethod
    def generate_signature(
        secret: str,
        method: str,
        path: str,
        x_date: str,
        content_md5: str = ""
    ):
        string_to_sign = (
            f"x-date: {x_date}\n"
            f"{method.upper()}\n"
            f"application/json\n"
            f"application/json\n"
            f"{content_md5}\n"
            f"{path}"
        )
        signature = hmac.new(secret.encode("utf-8"), string_to_sign.encode("utf-8"), hashlib.sha1).digest()
        return base64.b64encode(signature).decode("utf-8")

class PuduApiClient:

    BASE_URL = ("https://css-open-platform.pudutech.com") # /pudu-entry" # "https://css-open-platform.pudutech.com"
    
    def __init__(
        self,
        app_key: str,
        app_secret: str,
        timeout=30
    ):
        self.app_key = app_key
        self.app_secret = app_secret
        self.timeout = timeout

    def _build_path(
        self,
        endpoint,
        params=None
    ):
        if not params:
            return endpoint
        sorted_params = sorted(params.items(), key=lambda x: x[0])               
        query = "&".join(f"{k}={v}" for k, v in sorted_params)        
        #query = unquote(urlencode(sorted_params, doseq=True))
        return f"{endpoint}?{query}"

    def _headers(
        self,
        method,
        path,
        body=""
    ):
        # print(path) # 為了看 open_map url path 的差別
        x_date = PuduAuth.utc_date()
        content_md5 = ""
        if method.upper() == "POST":
            if body:
                content_md5 = (PuduAuth.md5_base64(body))
        signature = (
            PuduAuth.generate_signature(
                self.app_secret,
                method,
                path,
                x_date,
                content_md5
            )
        )
        authorization = (
            f'hmac id="{self.app_key}", '
            f'algorithm="hmac-sha1", '
            f'headers="x-date", '
            f'signature="{signature}"'
        )
        headers = {
            "Accept": "application/json",
            "Content-Type": "application/json",
            "x-date": x_date,
            "Authorization": authorization
        }
        if content_md5:
            headers["Content-MD5"] = content_md5
        return headers

    def _get_v2(self, endpoint, params=None):
        # ✅ 簽名用「原始未編碼」的 query 字串（依 key 排序，與成功 API 一致）
        if params:
            sorted_items = sorted(params.items(), key=lambda x: x[0])
            raw_query = "&".join(f"{k}={v}" for k, v in sorted_items)
            sign_path = f"{endpoint}?{raw_query}"
        else:
            sign_path = endpoint

        headers = self._headers("GET", sign_path)
        print(headers)
        url = self.BASE_URL + endpoint

        try:
            r = requests.get(
                url,
                params=params,        # ✅ 送出時才由 requests 編碼
                headers=headers,
                timeout=self.timeout
            )
            print("\n===== DEBUG =====")
            print("SIGN PATH:", sign_path)
            print("REAL URL :", r.url)
            print("STATUS   :", r.status_code)
            print("TEXT     :", r.text)
            r.raise_for_status()
            return r.json()
        except Exception as err:
            print(f"\n❌ API 請求失敗，錯誤原因: {err}")
            return {"success": False, "error_msg": str(err), "data": []}

    def _get(
        self,
        endpoint,
        params=None
    ):
        path = self._build_path(endpoint, params)
        headers = self._headers("GET", path)        
        url = self.BASE_URL + endpoint

        try:
            r = requests.get(
                url,
                params=params,
                headers=headers,
                timeout=self.timeout
            )
            r.raise_for_status()
            return r.json()
        except Exception as err:
            print(f"\n❌ API 請求失敗，錯誤原因: {err}")
            return {
                "success": False, 
                "error_msg": str(err), 
                "data": []  # 👈 塞一個空列表防呆，確保後面調用 .get('data') 不會崩潰
            }

    def _post(self, endpoint, payload):
        body = json.dumps(
            payload,
            ensure_ascii=False,
            separators=(",", ":")
        )

        headers = self._headers(
            "POST",
            endpoint,
            body
        )

        url = self.BASE_URL + endpoint

        try:
            r = requests.post(
                url,
                data=body.encode("utf-8"),
                headers=headers,
                timeout=self.timeout
            )

            # 🔥 先印出 response（超重要）
            print("\n=== POST DEBUG ===")
            print("URL:", url)
            print("BODY:", body)
            print("HEADERS:", headers)
            print("STATUS:", r.status_code)
            print("RESPONSE:", r.text)

            r.raise_for_status()

            return r.json()

        except Exception as err:
            print(f"\n❌ POST API 請求失敗: {err}")

            return {
                "success": False,
                "error_msg": str(err),
                "status_code": getattr(r, "status_code", None) if 'r' in locals() else None,
                "response": r.text if 'r' in locals() else None,  # 🔥 超關鍵
                "data": []
            }


    def _post_v1(
        self,
        endpoint,
        payload
    ):
        body = json.dumps(
            payload,
            ensure_ascii=False,
            separators=(",", ":")
        )
        headers = self._headers(
            "POST",
            endpoint,
            body
        )
        url = self.BASE_URL + endpoint
        r = requests.post(
            url,
            data=body.encode("utf-8"),
            headers=headers,
            timeout=self.timeout
        )
        r.raise_for_status()
        return r.json()

    def get_shops(
        self,
        limit=100,
        offset=0
    ):
        return self._get(
            "/pudu-entry/data-open-platform-service/v1/api/shop",
            {
                "limit": limit,
                "offset": offset
            }
        )

    def get_points(
        self,
        sn,
        limit=100,
        offset=0,
    ):
        return self._get(
            "/pudu-entry/map-service/v1/open/point",
            {
                "sn": sn,
                "limit": limit,
                "offset": offset
            }
        )

    def get_robots(
        self,
        shop_id
    ):
        return self._get(
            "/pudu-entry/data-open-platform-service/v1/api/robot",
            {
                "shop_id": shop_id
            }
        )

    def get_maps(
        self,
        shop_id
    ):
        return self._get(
            "/pudu-entry/data-open-platform-service/v1/api/maps",
            {
                "shop_id": shop_id
            }
        )

    def get_map_detail(
        self,
        shop_id,
        map_name,
        width=1200,
        height=800
    ):
        return self._get(
            "/pudu-entry/data-open-platform-service/v1/api/map",
            {
                "shop_id": shop_id,
                "map_name": map_name,
                "device_width": width,
                "device_height": height
            }
        )

    def get_robot_position(
        self,custom_content
        shop_id,
        sn
    ):
        return self._get(
            "/pudu-entry/data-open-platform-service/v1/api/map/robotCurrentPosition",
            {
                "shop_id": shop_id,
                "sn": sn
            }
        )

    def get_position(
        self,        
        sn
    ):
        return self._get(
            "/pudu-entry/open-platform-service/v1/robot/get_position",
            {                
                "sn": sn,
            }
        )

    def open_map(
        self,
        shop_id,
        map_name        
    ):
        return self._get(
            "/pudu-entry/map-service/v1/open/map",
            {
                "shop_id": shop_id,                
                "map_name": map_name
            }
        )

    def get_voice_list(
        self,        
        sn
    ):
        return self._get(
            "/pudu-entry/open-platform-service/v1/voice/list",
            {                
                "sn": sn,
            }
        )

    def get_map_list(self, sn):
        return self._get(
            "/pudu-entry/map-service/v1/open/list",
            {
                "sn": sn
            }
        )

    def get_map_current(self, sn, need_element=False):
        return self._get(
            "/pudu-entry/map-service/v1/open/current",
            {
                "sn": sn,
                "need_element": need_element
            }
        )   
        
    def switch_map(
        self,
        sn,
        map_name,
        point,
        call_device_name="PythonSDK"
    ):
        return self._post(
            "/pudu-entry/open-platform-service/v1/switch_map",
            {
                "sn": sn,
                "map_name": map_name,
                "point": point,
                "call_device_name": call_device_name
            }
        )

    def switch_map_in_elevator(
        self,
        sn,
        map_name,
        point,
        call_device_name="PythonSDK"
    ):
        return self._post(
            "/pudu-entry/open-platform-service/v1/robot/map/switch_in_elevator",
            {
                "sn": sn,
                "map_name": map_name,
                "point": point,
                "call_device_name": call_device_name
            }
        )

    def create_transport_task(self, payload):
        return self._post(
            "/pudu-entry/open-platform-service/v1/transport_task",
            payload
        )

    def create_delivery_task(self, payload):
        return self._post(
            "/pudu-entry/open-platform-service/v1/delivery_task",
            payload
        )

    def get_robot_current_position(self, shop_id, sn):
        return self._get(
            "/pudu-entry/data-open-platform-service/v1/api/map/robotCurrentPosition",
            {
                "shop_id": shop_id,
                "sn": sn
            }
        )
    
    def get_door_state(
        self,
        sn,
        ):
        return self._get(
            "/pudu-entry/open-platform-service/v1/door_state",
            {
                "sn": sn,
            }
        )

    def get_by_sn(self, sn):
        return self._get(
            "/pudu-entry/open-platform-service/v1/status/get_by_sn",
            {
                "sn": sn,
            } 
        )

    def get_by_sn2(self, sn):
        return self._get(
            "/pudu-entry/open-platform-service/v2/status/get_by_sn",
            {
                "sn": sn,
            }             
        )

    def state_get(self, sn):
        return self._get(
            "/pudu-entry/open-platform-service/v1/robot/task/state/get",
            {
                "sn": sn,
            }            
        )

    def call_list(self, sn):
        return self._get(
            "/pudu-entry/open-platform-service/v1/call/list",
            {
                "sn": sn,
            }             
        )

    def recharge(self, sn):
        return self._get(
            "/pudu-entry/open-platform-service/v2/recharge",
            {
                "sn": sn,
            }            
        )

    def cancel_task(self, payload):
        return self._post(
            "/pudu-entry/open-platform-service/v1/cancel_task",
            payload
        )

    def screen_set(self, payload):
        return self._post(
            "/pudu-entry/open-platform-service/v1/robot/screen/set",
            payload
        )

    def custom_call_cancel(self, payload):
        return self._post(
            "/pudu-entry/open-platform-service/v1/custom_call/cancel",
            payload
        )

    def control_doors(self, payload):
        return self._post(
            "/pudu-entry/open-platform-service/v1/control_doors",
            payload
        )

    def custom_content(self, payload):
        return self._post(
            "/pudu-entry/open-platform-service/v1/custom_content",
            payload
        )

    def custom_complete(self, payload):
        return self._post(
            "/pudu-entry/open-platform-service/v1/custom_call/complete",
            payload
        )    

    def custom_call2(self, payload):
        return self._post(
            "/pudu-entry/open-platform-service/v1/custom_call",
            payload
        )

    def custom_call(
        self,
        sn,
        shop_id,
        map_name,
        point,
        point_type="table",
        call_device_name="PythonSDK",
        call_mode="IMG",
        mode_data=None,
        do_not_queue=False,
        robot_group_ids=None,
        filter_category_ids=None,
        priority=1
    ):
        payload = {
            "sn": sn,
            "shop_id": shop_id,
            "map_name": map_name,
            "point": point,
            "point_type": point_type,
            "call_device_name": call_device_name,
            "call_mode": call_mode,
            "mode_data": mode_data or {},
            "do_not_queue": do_not_queue,
            "robot_group_ids": robot_group_ids or [],
            "filter_category_ids": filter_category_ids or [],
            "priority": priority
        }

        return self._post(
            "/pudu-entry/open-platform-service/v1/custom_call",
            payload
        )
    

In [3]:
client = PuduApiClient(app_key=pudu_api_key, app_secret=pudu_api_secret)
print(f"PuduApiClient initialized.  {client}")

PuduApiClient initialized.  <__main__.PuduApiClient object at 0x000001FB0D20C6E0>


In [4]:
map_data = client.open_map(shop_id=408250001,map_name='1#1#內湖展間v20')
print(json.dumps(map_data, indent=2, ensure_ascii=False))

{
  "data": {
    "element_list": [
      {
        "clean_path_list": [],
        "id": "倉庫位",
        "mode": "table",
        "name": "倉庫位",
        "type": "source",
        "vector_list": [
          -12.01,
          -7.62,
          1.56
        ]
      },
      {
        "clean_path_list": [],
        "id": "喵喵充電",
        "mode": "table",
        "name": "喵喵充電",
        "type": "source",
        "vector_list": [
          -5.38,
          -7.25,
          -1.58
        ]
      },
      {
        "clean_path_list": [],
        "id": "喵喵待機",
        "mode": "dining_outlet",
        "name": "喵喵待機",
        "type": "source",
        "vector_list": [
          -2.33,
          -2.94,
          1.56
        ]
      },
      {
        "clean_path_list": [],
        "id": "堆棧輸送帶",
        "mode": "table",
        "name": "堆棧輸送帶",
        "type": "source",
        "vector_list": [
          -12.48,
          0.12,
          3.14
        ]
      },
      {
        "clean_path_list": [],

In [5]:
print(map_data['data']['url'])

https://sg-tech-cloud-open.s3.amazonaws.com/pudu_cloud_platform/production/map_image/1012483295230886144_optemap.png


In [8]:
robot_sn = Flashbot # BellabotPro does not support
result = client.recharge(sn=robot_sn)
print(result)

{'data': {'desc': 'success', 'task_id': '8b2fed004c874dfdacd5a9309a86faf0'}, 'message': 'SUCCESS', 'trace_id': 'APIDKWT5TRzAOgyuVr2ERhbY86J0OgS8cb8k6J7J_4de7a920-9f80-4ece-811d-e7323048e07c'}


In [5]:
robot_sn = Flashbot # BellabotPro
result = client.get_door_state(sn=robot_sn)
print("Door State:", json.dumps(result, indent=2, ensure_ascii=False))

Door State: {
  "data": {
    "door_states": [
      {
        "door_number": "H_01",
        "state": "OPENED"
      },
      {
        "door_number": "H_02",
        "state": "OPENED"
      },
      {
        "door_number": "H_03",
        "state": "CLOSED"
      },
      {
        "door_number": "H_04",
        "state": "CLOSED"
      }
    ],
    "door_template_id": 9
  },
  "message": "SUCCESS",
  "trace_id": "APIDKWT5TRzAOgyuVr2ERhbY86J0OgS8cb8k6J7J_c3acf65e-76dd-464c-b533-044c27001ae3"
}


In [8]:
robot_sn = Flashbot # BellabotPro
result = client.get_voice_list(sn=robot_sn)
print(json.dumps(result, indent=2, ensure_ascii=False))

{
  "data": {
    "voice_list": []
  },
  "message": "SUCCESS",
  "trace_id": "APIDKWT5TRzAOgyuVr2ERhbY86J0OgS8cb8k6J7J_60885a00-66fb-48d4-b379-80f5281149bc"
}


In [9]:
task_id = "1781659544535743"

payload={
  "sn": Flashbot,
  "is_auto_back" : False,
  "task_id": task_id,
}

payload={
  "sn": Flashbot,
  "is_auto_back" : False,
}

result = client.custom_call_cancel(payload)
print(result)


=== POST DEBUG ===
URL: https://css-open-platform.pudutech.com/pudu-entry/open-platform-service/v1/custom_call/cancel
BODY: {"sn":"8FF055923050007","is_auto_back":false}
HEADERS: {'Accept': 'application/json', 'Content-Type': 'application/json', 'x-date': 'Wed, 08 Jul 2026 07:30:36 GMT', 'Authorization': 'hmac id="APIDKWT5TRzAOgyuVr2ERhbY86J0OgS8cb8k6J7J", algorithm="hmac-sha1", headers="x-date", signature="8RQNuEpODL1x87yTa8ZPw7HXWDw="', 'Content-MD5': 'ZmVkMGUwYTY1MWEzZDBlNjFlMjhhNDFkMjZkOGUwZDI='}
STATUS: 200
RESPONSE: {"message":"SUCCESS","trace_id":"APIDKWT5TRzAOgyuVr2ERhbY86J0OgS8cb8k6J7J_061a382d-5173-4f0e-bc30-ab48829dd1fa"}
{'message': 'SUCCESS', 'trace_id': 'APIDKWT5TRzAOgyuVr2ERhbY86J0OgS8cb8k6J7J_061a382d-5173-4f0e-bc30-ab48829dd1fa'}


In [10]:
robot_sn = Flashbot
task_id = "1781659544535743" # "1781598615564146" #"1781592605134936"
payload={"task_id": task_id,}
result = client.custom_complete(payload)
print(json.dumps(result, indent=2, ensure_ascii=False))


=== POST DEBUG ===
URL: https://css-open-platform.pudutech.com/pudu-entry/open-platform-service/v1/custom_call/complete
BODY: {"task_id":"1781659544535743"}
HEADERS: {'Accept': 'application/json', 'Content-Type': 'application/json', 'x-date': 'Wed, 08 Jul 2026 07:30:56 GMT', 'Authorization': 'hmac id="APIDKWT5TRzAOgyuVr2ERhbY86J0OgS8cb8k6J7J", algorithm="hmac-sha1", headers="x-date", signature="VYLT7mAImH9jaZjpo5jvnOMhI9E="', 'Content-MD5': 'YTdiZGM0MWM2ZTliMWU2MDZmMTYyNmU4NjE4NjdlYTk='}
STATUS: 500
RESPONSE: {"message":"CLOUD_OPEN_TASK_STATUS_ERROR","trace_id":"APIDKWT5TRzAOgyuVr2ERhbY86J0OgS8cb8k6J7J_bd284ba8-ffd0-4489-862c-827d1bebc5df"}

❌ POST API 請求失敗: 500 Server Error: Internal Server Error for url: https://css-open-platform.pudutech.com/pudu-entry/open-platform-service/v1/custom_call/complete
{
  "success": false,
  "error_msg": "500 Server Error: Internal Server Error for url: https://css-open-platform.pudutech.com/pudu-entry/open-platform-service/v1/custom_call/complete",
  

In [11]:
robot_sn = Flashbot

payload={
  "sn": Flashbot,
  "payload": {
    "tasks": [
      {
        "name": "喵喵待機",
        "type": "Call"
      }
    ]
  }
}

result = client.cancel_task(payload)
print(json.dumps(result, indent=2, ensure_ascii=False))


=== POST DEBUG ===
URL: https://css-open-platform.pudutech.com/pudu-entry/open-platform-service/v1/cancel_task
BODY: {"sn":"8FF055923050007","payload":{"tasks":[{"name":"喵喵待機","type":"Call"}]}}
HEADERS: {'Accept': 'application/json', 'Content-Type': 'application/json', 'x-date': 'Wed, 08 Jul 2026 07:31:03 GMT', 'Authorization': 'hmac id="APIDKWT5TRzAOgyuVr2ERhbY86J0OgS8cb8k6J7J", algorithm="hmac-sha1", headers="x-date", signature="KcSYqsP4pReg/lB6Yb0vt1JtsnU="', 'Content-MD5': 'OTk1OTJjMGQ4NzQ5ZjMxMmZhNDhhODYyNjAwNzAyOGI='}
STATUS: 200
RESPONSE: {"data":{"code":11001,"data":{"tasks":[{"failure_code":11001,"name":"喵喵待機","success":false,"type":"Call"}]},"message":"task not exist"},"message":"SUCCESS","trace_id":"APIDKWT5TRzAOgyuVr2ERhbY86J0OgS8cb8k6J7J_89d26517-72e6-407e-835a-833f3aad51d9"}
{
  "data": {
    "code": 11001,
    "data": {
      "tasks": [
        {
          "failure_code": 11001,
          "name": "喵喵待機",
          "success": false,
          "type": "Call"
        }
   

In [31]:
robot_sn = Flashbot # BellabotPro

payload={
  "sn": Flashbot,
  "payload": {
    "info": {
      "content": "和椿科技自動化整合機器人解決方案",
      "show": False, # False 可以用來取消顯示
    }
  }
}

result = client.screen_set(payload)
print(json.dumps(result, indent=2, ensure_ascii=False))


=== POST DEBUG ===
URL: https://css-open-platform.pudutech.com/pudu-entry/open-platform-service/v1/robot/screen/set
BODY: {"sn":"8FF055923050007","payload":{"info":{"content":"和椿科技自動化整合機器人解決方案","show":false}}}
HEADERS: {'Accept': 'application/json', 'Content-Type': 'application/json', 'x-date': 'Thu, 02 Jul 2026 07:59:58 GMT', 'Authorization': 'hmac id="APIDKWT5TRzAOgyuVr2ERhbY86J0OgS8cb8k6J7J", algorithm="hmac-sha1", headers="x-date", signature="XyxWDTpJ3mpMWyr95hIl5nobhxU="', 'Content-MD5': 'OGU4MGNhYjdkZjQ2NDdiZDAzNTRmYTQ1MDQxMzgxZjI='}
STATUS: 200
RESPONSE: {"data":{"code":0,"message":""},"message":"SUCCESS","trace_id":"APIDKWT5TRzAOgyuVr2ERhbY86J0OgS8cb8k6J7J_f19bf6ae-a078-4fd8-9fee-3e0d6a69846c"}
{
  "data": {
    "code": 0,
    "message": ""
  },
  "message": "SUCCESS",
  "trace_id": "APIDKWT5TRzAOgyuVr2ERhbY86J0OgS8cb8k6J7J_f19bf6ae-a078-4fd8-9fee-3e0d6a69846c"
}


In [7]:
robot_sn = Flashbot # BellabotPro

payload1={
    "sn": "8FF055923050007",
    "payload": {
      "control_states": [
        { "operation": False, "door_number": "H_01" },
        { "operation": False, "door_number": "H_02" },
        { "operation": False, "door_number": "H_03" },
        { "operation": False, "door_number": "H_04" }
      ]
    }
  }

payload={
    "sn": "8FF055923050007",
    "payload": {
      "control_states": [
        { "operation": False, "door_number": "H_03" },
      ]
    }
  }

result = client.control_doors(payload1)
#print(result)
print(json.dumps(result, indent=2, ensure_ascii=False))


=== POST DEBUG ===
URL: https://css-open-platform.pudutech.com/pudu-entry/open-platform-service/v1/control_doors
BODY: {"sn":"8FF055923050007","payload":{"control_states":[{"operation":false,"door_number":"H_01"},{"operation":false,"door_number":"H_02"},{"operation":false,"door_number":"H_03"},{"operation":false,"door_number":"H_04"}]}}
HEADERS: {'Accept': 'application/json', 'Content-Type': 'application/json', 'x-date': 'Mon, 13 Jul 2026 05:55:25 GMT', 'Authorization': 'hmac id="APIDKWT5TRzAOgyuVr2ERhbY86J0OgS8cb8k6J7J", algorithm="hmac-sha1", headers="x-date", signature="uJLu8bFXu9rkD5yu+jnAyKA+wlY="', 'Content-MD5': 'Y2IwOTU3MjI2ODZlN2E5MTg5NmJiM2Y0MWI0MjdlNDA='}
STATUS: 200
RESPONSE: {"data":{"door_states":[{"door_number":"H_01","state":"CLOSED"},{"door_number":"H_02","state":"CLOSED"},{"door_number":"H_03","state":"CLOSED"},{"door_number":"H_04","state":"CLOSED"}],"door_template_id":0},"message":"SUCCESS","trace_id":"APIDKWT5TRzAOgyuVr2ERhbY86J0OgS8cb8k6J7J_497d9394-a658-4663-ab0

In [ ]:
robot_sn = Flashbot # BellabotPro

open_close = True

for i in range(10):

    payload={
        "sn": robot_sn,
        "payload": {
        "control_states": [
            { "operation": open_close, "door_number": "H_01" },
            { "operation": open_close, "door_number": "H_02" },
            { "operation": open_close, "door_number": "H_03" },
            { "operation": open_close, "door_number": "H_04" }
        ]
        }
    }

    result = client.control_doors(payload)
    print(result)
    #print(json.dumps(result, indent=2, ensure_ascii=False))
    open_close = not open_close
    time.sleep(7)



=== POST DEBUG ===
URL: https://css-open-platform.pudutech.com/pudu-entry/open-platform-service/v1/control_doors
BODY: {"sn":"8FF055923050007","payload":{"control_states":[{"operation":true,"door_number":"H_01"},{"operation":true,"door_number":"H_02"},{"operation":true,"door_number":"H_03"},{"operation":true,"door_number":"H_04"}]}}
HEADERS: {'Accept': 'application/json', 'Content-Type': 'application/json', 'x-date': 'Thu, 11 Jun 2026 06:51:37 GMT', 'Authorization': 'hmac id="APIDKWT5TRzAOgyuVr2ERhbY86J0OgS8cb8k6J7J", algorithm="hmac-sha1", headers="x-date", signature="bntNTelVVYlK0oUJWzueyOAdbO8="', 'Content-MD5': 'ZTViNzI2YmJjMGQ0MDNmNDMwNjlmMGU4MWQ4NzQyNTI='}
STATUS: 200
RESPONSE: {"data":{"door_states":[{"door_number":"H_01","state":"OPENED"},{"door_number":"H_02","state":"OPENED"},{"door_number":"H_03","state":"OPENED"},{"door_number":"H_04","state":"OPENED"}],"door_template_id":0},"message":"SUCCESS","trace_id":"APIDKWT5TRzAOgyuVr2ERhbY86J0OgS8cb8k6J7J_73587186-fc68-4cee-850f-df

In [17]:
robot_sn = Flashbot # BellabotPro
result = client.get_points(sn=robot_sn,limit=100,offset=0)
#print(result)
#print("Robot Position:", json.dumps(result, indent=2, ensure_ascii=False))
points = result.get('data').get('list')
for each_point in points:
    print(each_point['name'])


倉庫位
喵喵充電
喵喵待機
堆棧輸送帶
托托待機
產線位
豆豆待機
轉轉待機
閃閃待機
高高待機
乘梯點
待梯點


In [10]:
robot_sn = Flashbot #BellabotPro
result = client.get_position(sn=robot_sn)
print(json.dumps(result, indent=2, ensure_ascii=False))

{
  "data": {
    "floor": "0",
    "map_name": "1#1#內湖展間v20",
    "position": {
      "x": -11.00687843964324,
      "y": -8.220164562779578,
      "z": -1.5906676288440886
    }
  },
  "message": "SUCCESS",
  "trace_id": "APIDKWT5TRzAOgyuVr2ERhbY86J0OgS8cb8k6J7J_23610c8d-3745-4ee2-ad47-8f3dc4f60273"
}


In [11]:
print("Getting shops...")
shops = client.get_shops(limit=10,offset=0)
all_shops = shops.get('data').get('list')
for each_shop in all_shops:
        print(each_shop['shop_id'],each_shop['shop_name'])

Getting shops...
22753 鸿匠
14060002 九合兰园休闲农场
14130244 屋馬 國安店
14210009 屋馬 崇德店
14210011 屋馬 中友店
14430019 T001247-碳佐麻里時代店
14610148 HerdarTech
14810056 貝蛤蛤海鮮餐酒館-中壢店（前綠澀）
14830056 季河-台中辦公室
149418420 任務咖啡-台中店


In [12]:
# Cell 4: Get Robots for a specific shop
# Use a hardcoded shop_id or the one extracted from the previous cell if available
robots = client.get_robots(shop_id=Aurotek_id)
#print("Robots:", json.dumps(robots, indent=2, ensure_ascii=False))
if robots and robots.get('data'):
    if len(robots['data']['list']) > 0:
        for each_robot in robots['data']['list']:
            print(each_robot)
            print(f"Robot序號: {each_robot['sn']}, 公司名稱: {each_robot['shop_name']}")
    else:
        print("No robot SN found in the first robot entry.")
else:
    print("No robots data returned or robots list is empty.")

{'first_active_time': 1772781310, 'mac': '50:CF:14:0D:76:87', 'nickname': 'T600潛伏', 'product_code': 'PuduT600Underride', 'shop_id': '408250001', 'shop_name': '和椿科技內湖總部', 'sn': '83A025B13070005'}
Robot序號: 83A025B13070005, 公司名稱: 和椿科技內湖總部
{'first_active_time': 1763600889, 'mac': '50:CF:14:0D:6F:3C', 'nickname': '咻咻', 'product_code': 'MT1Vac', 'shop_id': '408250001', 'shop_name': '和椿科技內湖總部', 'sn': '899025912070010'}
Robot序號: 899025912070010, 公司名稱: 和椿科技內湖總部
{'first_active_time': 1763609982, 'mac': '50:CF:14:0D:6D:CD', 'nickname': '聰聰', 'product_code': 'CC1Pro', 'shop_id': '408250001', 'shop_name': '和椿科技內湖總部', 'sn': '888015806070013'}
Robot序號: 888015806070013, 公司名稱: 和椿科技內湖總部
{'first_active_time': 1781782035, 'mac': '68:24:99:F8:8F:9A', 'nickname': 'BG1', 'product_code': 'BG1Pro', 'shop_id': '408250001', 'shop_name': '和椿科技內湖總部', 'sn': '858096413080017'}
Robot序號: 858096413080017, 公司名稱: 和椿科技內湖總部
{'first_active_time': 1724034857, 'mac': '08:E9:F6:CF:0E:F4', 'nickname': '豆豆', 'product_code': 'Ket

In [13]:
# Cell 5: Get Robot Position
# Use the shop_id and sn, falling back to hardcoded values if not dynamically set
shop_id_to_use = Aurotek_id
sn_to_use = BellabotPro # robots['data']['list'][1].get('sn') # "8BR054C02050014"

print(f"Getting position for robot SN: {sn_to_use} in shop ID: {shop_id_to_use}...")
position = client.get_robot_position(shop_id=shop_id_to_use, sn=sn_to_use)
print("Robot Position:", json.dumps(position, indent=2, ensure_ascii=False))

Getting position for robot SN: 8BR054C02050014 in shop ID: 408250001...
Robot Position: {
  "data": {
    "floor": "1",
    "map_name": "1#1#內湖展間v20",
    "position": {
      "x": -106.0623195723457,
      "y": 142.0085245028261,
      "z": -1.8298252890224462
    }
  },
  "message": "ok",
  "trace_id": "APIDKWT5TRzAOgyuVr2ERhbY86J0OgS8cb8k6J7J_f44c1c99-876b-4b00-b9df-7b54c87ca938"
}


In [ ]:
position = client.get_robot_position(shop_id=Aurotek_id, sn=BellabotPro)
print("Robot Position:", json.dumps(position, indent=2, ensure_ascii=False))

Robot Position: {
  "data": {
    "floor": "1",
    "map_name": "1#1#內湖展間v20",
    "position": {
      "x": -103.20991111042538,
      "y": 47.61198421830269,
      "z": 1.4460216665961345
    }
  },
  "message": "ok",
  "trace_id": "APIDKWT5TRzAOgyuVr2ERhbY86J0OgS8cb8k6J7J_4b064520-2349-475e-83a0-2aef64ff77e0"
}


In [112]:
result = client.get_map_list(sn=Flashbot)
#result = client.get_map_current(sn=sn_to_use)
print(f"sn : {Flashbot}, Map list Result:", json.dumps(result, indent=2, ensure_ascii=False))


===== DEBUG =====
SIGN PATH: /pudu-entry/map-service/v1/open/list?sn=8FF055923050007
REAL URL : https://css-open-platform.pudutech.com/pudu-entry/map-service/v1/open/list?sn=8FF055923050007
STATUS   : 200
TEXT     : {"data":{"list":[{"floor":"1","name":"1#1#內湖展間v20"}],"shop_id":408250001},"message":"SUCCESS","trace_id":"APIDKWT5TRzAOgyuVr2ERhbY86J0OgS8cb8k6J7J_be3e2bc5-c546-4107-ad21-cfcd719ee1bf"}
sn : 8FF055923050007, Map list Result: {
  "data": {
    "list": [
      {
        "floor": "1",
        "name": "1#1#內湖展間v20"
      }
    ],
    "shop_id": 408250001
  },
  "message": "SUCCESS",
  "trace_id": "APIDKWT5TRzAOgyuVr2ERhbY86J0OgS8cb8k6J7J_be3e2bc5-c546-4107-ad21-cfcd719ee1bf"
}


In [ ]:
for i in range(len(robots['data']['list'])):
    sn_to_use = robots['data']['list'][i].get('sn')
    result = client.get_map_list(sn=sn_to_use)
    #result = client.get_map_current(sn=sn_to_use)
    print(f"{i} : , sn : {sn_to_use}, Map list Result:", json.dumps(result, indent=2, ensure_ascii=False))

0 : , sn : 83A025B13070005, Map list Result: {
  "data": {
    "list": [
      {
        "floor": "0",
        "name": "0#0#內湖展間v20測試"
      }
    ]
  },
  "message": "SUCCESS",
  "trace_id": "APIDKWT5TRzAOgyuVr2ERhbY86J0OgS8cb8k6J7J_93c84df1-5f64-4a1b-842a-e344afff20cf"
}
1 : , sn : 899025912070010, Map list Result: {
  "data": {
    "list": [
      {
        "floor": "0",
        "name": "0#0#內湖展間v20副本測試"
      },
      {
        "floor": "1",
        "name": "1#12#碼頭"
      },
      {
        "floor": "1",
        "name": "1#8#測試 內湖展間清潔v4 測試"
      },
      {
        "floor": "0",
        "name": "0#0#內湖展間v20測試"
      },
      {
        "floor": "1",
        "name": "1#9#內湖展間3D停車場測試"
      },
      {
        "floor": "1",
        "name": "1#8#行銷影片MT1VAC"
      },
      {
        "floor": "1",
        "name": "1#7#內湖展間清潔v4"
      },
      {
        "floor": "0",
        "name": "0#0#內湖展間v20行銷影片"
      },
      {
        "floor": "1",
        "name": "1#3#內湖展間清潔v2"
      },
      {
  

In [ ]:
for i in range(len(robots['data']['list'])):
    sn_to_use = robots['data']['list'][i].get('sn')
    result = client.get_map_list(sn=sn_to_use)
    maps = result.get('data').get('list')
    print(f"{i} : , sn : {sn_to_use}, Map list Result:")
    for each_map in maps:
        print(each_map.get('name'))

0 : , sn : 83A025B13070005, Map list Result:
0#0#內湖展間v20測試
1 : , sn : 899025912070010, Map list Result:
0#0#內湖展間v20副本測試
1#12#碼頭
1#8#測試 內湖展間清潔v4 測試
0#0#內湖展間v20測試
1#9#內湖展間3D停車場測試
1#8#行銷影片MT1VAC
1#7#內湖展間清潔v4
0#0#內湖展間v20行銷影片
1#3#內湖展間清潔v2
1#1#內湖展間v20
1#5#內湖1樓戶外
1#1#內湖展間v17
2#0#內湖展間清潔v1
1#1#內湖展間v16
內湖展間v16
0#0#和椿2F辦公室測試
1#1#內湖展間清潔v1
2 : , sn : 888015806070013, Map list Result:
0#0#內湖展間v20副本測試
1#12#碼頭
1#8#測試 內湖展間清潔v4 測試
0#0#內湖展間v20測試
1#9#內湖展間3D停車場測試
1#8#行銷影片MT1VAC
1#7#內湖展間清潔v4
0#0#內湖展間v20行銷影片
1#3#內湖展間清潔v2
1#1#內湖展間v20
1#5#內湖1樓戶外
1#1#內湖展間v17
2#0#內湖展間清潔v1
1#1#內湖展間v16
內湖展間v16
0#0#和椿2F辦公室測試
1#1#內湖展間清潔v1
3 : , sn : 8PP044808050001, Map list Result:
0#0#內湖展間v20副本測試
1#12#碼頭
1#8#測試 內湖展間清潔v4 測試
0#0#內湖展間v20測試
1#9#內湖展間3D停車場測試
1#8#行銷影片MT1VAC
1#7#內湖展間清潔v4
0#0#內湖展間v20行銷影片
1#3#內湖展間清潔v2
1#1#內湖展間v20
1#5#內湖1樓戶外
1#1#內湖展間v17
2#0#內湖展間清潔v1
1#1#內湖展間v16
內湖展間v16
0#0#和椿2F辦公室測試
1#1#內湖展間清潔v1
4 : , sn : 8BR054C02050014, Map list Result:
1#1#內湖展間v20
5 : , sn : 826094717050002, Map list Result:
1#1#內湖展間v20
6 : , sn : 868025

In [ ]:
robot_sn = Flashbot #BellabotPro
result = client.get_map_list(sn=robot_sn) #BellabotPro
print(f"sn : {robot_sn}, Map list Result:", json.dumps(result, indent=2, ensure_ascii=False))

sn : 8BR054C02050014, Map list Result: {
  "data": {
    "list": [
      {
        "floor": "1",
        "name": "1#1#內湖展間v20"
      }
    ],
    "shop_id": 408250001
  },
  "message": "SUCCESS",
  "trace_id": "APIDKWT5TRzAOgyuVr2ERhbY86J0OgS8cb8k6J7J_3231cd3b-4568-42f1-b3ef-a5db9e330748"
}


In [11]:
# Cell 6: Switch Map
# Use the sn from previous cells or a hardcoded one
sn_to_use = Flashbot#BellabotPro
map_name_to_use = "1#1#內湖展間v20" # Example map name
point_to_use = "托托待機" # Example point

print(f"Switching map for robot SN: {sn_to_use} to map: {map_name_to_use} at point: {point_to_use}...")
result = client.switch_map(sn=sn_to_use, map_name=map_name_to_use, point=point_to_use)
print("Switch Map Result:", json.dumps(result, indent=2, ensure_ascii=False))

Switching map for robot SN: 8FF055923050007 to map: 1#1#內湖展間v20 at point: 托托待機...

=== POST DEBUG ===
URL: https://css-open-platform.pudutech.com/pudu-entry/open-platform-service/v1/switch_map
BODY: {"sn":"8FF055923050007","map_name":"1#1#內湖展間v20","point":"托托待機","call_device_name":"PythonSDK"}
HEADERS: {'Accept': 'application/json', 'Content-Type': 'application/json', 'x-date': 'Mon, 22 Jun 2026 02:39:29 GMT', 'Authorization': 'hmac id="APIDKWT5TRzAOgyuVr2ERhbY86J0OgS8cb8k6J7J", algorithm="hmac-sha1", headers="x-date", signature="zvkJcr6lQzGxzMV1Y3OL8Luuyc0="', 'Content-MD5': 'MzUzZDM1ZDEwMjNiZjE2M2IxODYzOGZkZDY5OTkwODE='}
STATUS: 500
RESPONSE: {"message":"CLOUD_OPEN_ROBOT_PARAM_ERROR","trace_id":"APIDKWT5TRzAOgyuVr2ERhbY86J0OgS8cb8k6J7J_2d316a87-d5b1-4c69-a2a6-5d3c206dc524"}

❌ POST API 請求失敗: 500 Server Error: Internal Server Error for url: https://css-open-platform.pudutech.com/pudu-entry/open-platform-service/v1/switch_map
Switch Map Result: {
  "success": false,
  "error_msg": "500

In [25]:
result = client.get_position(sn=BellabotPro)
print(result)
#print(json.dumps(result, indent=2, ensure_ascii=False))

{'data': {'floor': '1', 'map_name': '1#1#內湖展間v20', 'position': {'x': -5.567922550070222, 'y': -7.193878858198133, 'z': 3.116850114528708}}, 'message': 'SUCCESS', 'trace_id': 'APIDKWT5TRzAOgyuVr2ERhbY86J0OgS8cb8k6J7J_7225a9d4-0ec8-4d07-a3e1-1fbbff49a446'}


In [ ]:
result = client.get_map_current(sn=BellabotPro, need_element=True)
print(f"{i} : , sn : {sn_to_use}, Map list Result:", json.dumps(result, indent=2, ensure_ascii=False))

9 : , sn : 826094913050022, Map list Result: {
  "data": {
    "elements": [
      {
        "group": "",
        "id": "倉庫位",
        "max_speed": 0,
        "mode": "table",
        "name": "倉庫位",
        "type": "source",
        "vector": [
          -12.01,
          -7.62,
          1.56
        ],
        "width": 0
      },
      {
        "group": "",
        "id": "喵喵充電",
        "max_speed": 0,
        "mode": "table",
        "name": "喵喵充電",
        "type": "source",
        "vector": [
          -5.38,
          -7.25,
          -1.58
        ],
        "width": 0
      },
      {
        "group": "喵喵待機",
        "id": "喵喵待機",
        "max_speed": 0,
        "mode": "dining_outlet",
        "name": "喵喵待機",
        "type": "source",
        "vector": [
          -2.33,
          -2.94,
          1.56
        ],
        "width": 0
      },
      {
        "group": "",
        "id": "堆棧輸送帶",
        "max_speed": 0,
        "mode": "table",
        "name": "堆棧輸送帶",
        "ty

In [ ]:
result = client.get_robot_current_position(shop_id=Aurotek_id, sn=BellabotPro)
print(result)

{'data': {'floor': '1', 'map_name': '1#1#內湖展間v20', 'position': {'x': -103.20760486368856, 'y': 47.5786692983462, 'z': 1.4462970447786778}}, 'message': 'ok', 'trace_id': 'APIDKWT5TRzAOgyuVr2ERhbY86J0OgS8cb8k6J7J_6e9ecc56-c526-45c5-bc85-dadcfb489ceb'}


In [26]:
payload={
  "sn": Flashbot, #BellabotPro,
  "shop_id": Aurotek_id,
  "map_name": "1#1#內湖展間v20",
  "point": "高高待機",
  "point_type": "table",
  "call_mode": "CALL",
  "mode_data": {"text": "你好，歡迎光臨"}
}

result = client.custom_call2(payload)
print(result)


=== POST DEBUG ===
URL: https://css-open-platform.pudutech.com/pudu-entry/open-platform-service/v1/custom_call
BODY: {"sn":"8FF055923050007","shop_id":"408250001","map_name":"1#1#內湖展間v20","point":"高高待機","point_type":"table","call_mode":"CALL","mode_data":{"text":"你好，歡迎光臨"}}
HEADERS: {'Accept': 'application/json', 'Content-Type': 'application/json', 'x-date': 'Fri, 03 Jul 2026 03:24:18 GMT', 'Authorization': 'hmac id="APIDKWT5TRzAOgyuVr2ERhbY86J0OgS8cb8k6J7J", algorithm="hmac-sha1", headers="x-date", signature="EV2gdaBnIQy6HPwQzGCWgamzLGQ="', 'Content-MD5': 'NmRmMTE2NTUxNGI0ODcxM2ViOGQ4NjhhMmVmYmY1MDk='}
STATUS: 200
RESPONSE: {"data":{"queue":0,"remark":"","state":"CALL_SUCCESS","task_id":"1783049059959269"},"message":"SUCCESS","trace_id":"APIDKWT5TRzAOgyuVr2ERhbY86J0OgS8cb8k6J7J_69df0d9f-7cf3-49ed-ae63-42ff5add47f3"}
{'data': {'queue': 0, 'remark': '', 'state': 'CALL_SUCCESS', 'task_id': '1783049059959269'}, 'message': 'SUCCESS', 'trace_id': 'APIDKWT5TRzAOgyuVr2ERhbY86J0OgS8cb8k6J7J_6

In [22]:
pic_1 = "https://www.grandsys.com.tw/uploads/product/ASR/STTw_01.jpg"
pic_2 = "https://www.grandsys.com.tw/uploads/product/ASR/STTw_02.jpg"
pic_3 = "https://d3gjxtgqyywct8.cloudfront.net/o2o/image/02916822-2cc0-426c-be87-d0d4eb9d20b1.jpg"
pic_4 = "https://img2.91mai.com/o2o/image/9c40122f-f06a-49e4-ad0d-2965bb2e935c.jpg"

textstr = "霹靂車，尖端科技的結晶，是一部人性化的萬能電腦車。出現在我們這個無其不有的世界­，刀槍不入，無所不能。霹靂遊俠李麥克，充滿正義感，是一個英勇的自由鬥士。他以­無比的勇氣，超人的智慧，打擊犯罪，拯救善良無助的受害者。" 
textstr = "霹靂車，尖端科技的結晶，­刀槍不入，無所不能，是一部人性化的萬能電腦車，出現在我們這個無其不有的世界上。"

yturl = "https://www.youtube.com/watch?v=bPt4wpDXALM"
videourl = "https://www.pexels.com/zh-tw/download/video/8087317/"

In [ ]:
payload={
  "sn": Flashbot, # BellabotPro,
  "shop_id": Aurotek_id,
  "map_name": "1#1#內湖展間v20",
  "point": "喵喵待機", # "托托待機",
  "point_type": "table",
  "call_mode": "IMG",
  "mode_data": {
    "urls": [
      pic_3,
      pic_4
    ],
    "text": textstr,
    "switch_time": 5,
    "cancel_btn_time": 60,
    "show_timeout": 120
  }
}

payload2={
  "sn": Flashbot,
  "shop_id": Aurotek_id,
  "map_name": "1#1#內湖展間v20",
  "point": "喵喵待機", # "托托待機",
  "point_type": "table",
  "call_mode": "CALL",
  "mode_data": {"text": textstr},
}

payload3={
  "sn": Flashbot, #BellabotPro,
  "shop_id": Aurotek_id,
  "map_name": "1#1#內湖展間v20",
  "point": "托托待機",#"喵喵待機", # 
  "point_type": "table",
  "call_mode": "QR_CODE",
  "mode_data": {
      "qrcode": yturl,
      "text": textstr,
      "switch_time": 5,
      "cancel_btn_time": 10,
      "show_timeout": 30
      },
}

payload4={
  "sn": Flashbot,
  "shop_id": Aurotek_id,
  "map_name": "1#1#內湖展間v20",
  "point": "貓貓待機",# "喵喵待機", # "托托待機",# 
  "point_type": "table",
  "call_mode": "VIDEO",
  "mode_data": {
    "urls": [
      videourl
    ],
  }
}

#time.sleep(30)
# result = client.custom_call2(payload)
result = client.custom_call_cancel(payload4)
print(result)


=== POST DEBUG ===
URL: https://css-open-platform.pudutech.com/pudu-entry/open-platform-service/v1/custom_call/cancel
BODY: {"sn":"8FF055923050007","shop_id":"408250001","map_name":"1#1#內湖展間v20","point":"貓貓待機","point_type":"table","call_mode":"VIDEO","mode_data":{"urls":["https://www.pexels.com/zh-tw/download/video/8087317/"]}}
HEADERS: {'Accept': 'application/json', 'Content-Type': 'application/json', 'x-date': 'Thu, 02 Jul 2026 08:51:33 GMT', 'Authorization': 'hmac id="APIDKWT5TRzAOgyuVr2ERhbY86J0OgS8cb8k6J7J", algorithm="hmac-sha1", headers="x-date", signature="rK1/PWCHGST10EdJIJi5diZvdr8="', 'Content-MD5': 'YjRlMDU2MTAxOTZmZGE0NGUzNDI0ZWQ5YTc3MzY5YjM='}
STATUS: 200
RESPONSE: {"message":"SUCCESS","trace_id":"APIDKWT5TRzAOgyuVr2ERhbY86J0OgS8cb8k6J7J_932f1ae5-bdfe-487b-85ec-c934dc46c995"}
{'message': 'SUCCESS', 'trace_id': 'APIDKWT5TRzAOgyuVr2ERhbY86J0OgS8cb8k6J7J_932f1ae5-bdfe-487b-85ec-c934dc46c995'}


In [46]:
payload={
  "sn": Flashbot,  
  "point": "喵喵待機", 
  "point_type": "table",
  "call_mode": "QR_CODE",
  "mode_data": {
      "qrcode": yturl,      
      },
}

payload={
    "sn": "8FF055923050007",
    "map_name": "1#1#內湖展間v20",
    "point": "托托待機",
    "point_type": "table",    
    "call_mode": "QR_CODE",
    "mode_data": { "qrcode": yturl, "text": textstr},
    "do_not_queue": False,
    "call_device_name": "APIDKWT5TRzAOgyuVr2ERhbY86J0OgS8cb8k6J7J",
  }

#result = client.custom_call2(payload)
result = client.custom_call_cancel(payload)
print(result)


=== POST DEBUG ===
URL: https://css-open-platform.pudutech.com/pudu-entry/open-platform-service/v1/custom_call/cancel
BODY: {"sn":"8FF055923050007","map_name":"1#1#內湖展間v20","point":"托托待機","point_type":"table","call_mode":"QR_CODE","mode_data":{"qrcode":"https://www.youtube.com/watch?v=bPt4wpDXALM","text":"霹靂車，尖端科技的結晶，­刀槍不入，無所不能，是一部人性化的萬能電腦車，出現在我們這個無其不有的世界上。"},"do_not_queue":false,"call_device_name":"APIDKWT5TRzAOgyuVr2ERhbY86J0OgS8cb8k6J7J"}
HEADERS: {'Accept': 'application/json', 'Content-Type': 'application/json', 'x-date': 'Thu, 02 Jul 2026 08:14:51 GMT', 'Authorization': 'hmac id="APIDKWT5TRzAOgyuVr2ERhbY86J0OgS8cb8k6J7J", algorithm="hmac-sha1", headers="x-date", signature="6dynMYe0xrY8F9mSw1SwnWaRRq4="', 'Content-MD5': 'NGJjZDhhNjVmZWRlOGFhMzVkOGVhNDYwNGZhYjg5MjA='}
STATUS: 200
RESPONSE: {"message":"SUCCESS","trace_id":"APIDKWT5TRzAOgyuVr2ERhbY86J0OgS8cb8k6J7J_17f547c3-e2c7-47c4-845a-e1c4f8eaebd4"}
{'message': 'SUCCESS', 'trace_id': 'APIDKWT5TRzAOgyuVr2ERhbY86J0OgS8cb8k6J7J_17f54

In [42]:
result = client.custom_call(
        sn=BellabotPro,
        shop_id=Aurotek_id,
        map_name="1#1#內湖展間v20",
        point="托托待機",
        point_type="table",
        call_device_name="PythonSDK",
        call_mode="VIDEO",
        mode_data={"urls": [videourl],},
        do_not_queue=False,
        robot_group_ids=None,
        filter_category_ids=None,
        priority=1)

print(result)


=== POST DEBUG ===
URL: https://css-open-platform.pudutech.com/pudu-entry/open-platform-service/v1/custom_call
BODY: {"sn":"8BR054C02050014","shop_id":"408250001","map_name":"1#1#內湖展間v20","point":"托托待機","point_type":"table","call_device_name":"PythonSDK","call_mode":"VIDEO","mode_data":{"urls":["https://www.pexels.com/zh-tw/download/video/8087317/"]},"do_not_queue":false,"robot_group_ids":[],"filter_category_ids":[],"priority":1}
HEADERS: {'Accept': 'application/json', 'Content-Type': 'application/json', 'x-date': 'Thu, 02 Jul 2026 08:09:46 GMT', 'Authorization': 'hmac id="APIDKWT5TRzAOgyuVr2ERhbY86J0OgS8cb8k6J7J", algorithm="hmac-sha1", headers="x-date", signature="Kn+bf0LSO0z9RG4kWJ/DMj2Ct34="', 'Content-MD5': 'NTBmNTFmMGEwODBkZTJiNGMzMjUxMjFiNjA3YjE1NGE='}
STATUS: 200
RESPONSE: {"data":{"queue":0,"remark":"","state":"CALL_SUCCESS","task_id":"1782979787658154"},"message":"SUCCESS","trace_id":"APIDKWT5TRzAOgyuVr2ERhbY86J0OgS8cb8k6J7J_5d66ae8b-b49b-4c0d-9d00-17ee347191b6"}
{'data': {'

In [10]:
payload={
  "sn": Flashbot,
  "task_id": "1783933636159427"
}

result = client.custom_complete(payload)
print(result)


=== POST DEBUG ===
URL: https://css-open-platform.pudutech.com/pudu-entry/open-platform-service/v1/custom_call/cancel
BODY: {"sn":"8FF055923050007","task_id":"1783933636159427"}
HEADERS: {'Accept': 'application/json', 'Content-Type': 'application/json', 'x-date': 'Mon, 13 Jul 2026 09:10:38 GMT', 'Authorization': 'hmac id="APIDKWT5TRzAOgyuVr2ERhbY86J0OgS8cb8k6J7J", algorithm="hmac-sha1", headers="x-date", signature="gkNG2g+iHo+6geBQHywku0HM7zY="', 'Content-MD5': 'YjhlOWEzZjZlYzk0NzRkZTFjM2FlZmUxYzczOWJjYWU='}
STATUS: 500
RESPONSE: {"message":"CLOUD_OPEN_UNABLE_CANCEL_TASK","trace_id":"APIDKWT5TRzAOgyuVr2ERhbY86J0OgS8cb8k6J7J_5f1ebce4-9cba-4703-a5b5-76a65a9b7b45"}

❌ POST API 請求失敗: 500 Server Error: Internal Server Error for url: https://css-open-platform.pudutech.com/pudu-entry/open-platform-service/v1/custom_call/cancel
{'success': False, 'error_msg': '500 Server Error: Internal Server Error for url: https://css-open-platform.pudutech.com/pudu-entry/open-platform-service/v1/custom_cal

In [ ]:

time.sleep(3)
result = client.custom_call2(payload3)
print(result)
time.sleep(30)
result = client.custom_call_cancel(payload3)
result = client.custom_call2(payload4)
print(result)


=== POST DEBUG ===
URL: https://css-open-platform.pudutech.com/pudu-entry/open-platform-service/v1/custom_call
BODY: {"sn":"8FF055923050007","shop_id":"408250001","map_name":"1#1#內湖展間v20","point":"托托待機","point_type":"table","call_mode":"QR_CODE","mode_data":{"qrcode":"https://www.youtube.com/watch?v=bPt4wpDXALM","text":"霹靂車，尖端科技的結晶，­刀槍不入，無所不能，是一部人性化的萬能電腦車，出現在我們這個無其不有的世界上。","switch_time":5,"cancel_btn_time":10,"show_timeout":30}}
HEADERS: {'Accept': 'application/json', 'Content-Type': 'application/json', 'x-date': 'Thu, 02 Jul 2026 08:25:19 GMT', 'Authorization': 'hmac id="APIDKWT5TRzAOgyuVr2ERhbY86J0OgS8cb8k6J7J", algorithm="hmac-sha1", headers="x-date", signature="lIjTYkI26I8mLoAGuR0F2Cl0xQs="', 'Content-MD5': 'NTgxNjU1NjUxZjJjZmMxODY1NDdlYTkyODdkNTlmNGM='}
STATUS: 200
RESPONSE: {"data":{"queue":0,"remark":"","state":"CALL_SUCCESS","task_id":"1782980721153294"},"message":"SUCCESS","trace_id":"APIDKWT5TRzAOgyuVr2ERhbY86J0OgS8cb8k6J7J_ef47dd17-4e0c-467e-a0bd-a8d639533194"}
{'data': {'q

In [58]:
# 這是 custom call 的 payload 格式，custom content 不能用
payload5={
  "sn": Flashbot,
  "shop_id": 408250001, 
  "call_mode": "IMG",
  "mode_data": {
    "urls": [
      pic_1,
      pic_2
    ],
    "text": textstr,
    "switch_time": 5,
    "cancel_btn_time": 10,
    "show_timeout": 30
  }
}

payload6={
  "sn": Flashbot,
  "shop_id": 408250001, 
  "call_mode": "IMG",
  "mode_data": {
    "urls": [
      pic_3,
      pic_4
    ],
    "text": textstr,
    "switch_time": 5,
    "cancel_btn_time": 10,
    "show_timeout": 30
  }
}

time.sleep(3)
result = client.custom_content(payload5)
print(result)
time.sleep(5)
result = client.custom_content(payload6)
print(result)


=== POST DEBUG ===
URL: https://css-open-platform.pudutech.com/pudu-entry/open-platform-service/v1/custom_content
BODY: {"sn":"8FF055923050007","shop_id":408250001,"call_mode":"IMG","mode_data":{"urls":["https://www.grandsys.com.tw/uploads/product/ASR/STTw_01.jpg","https://www.grandsys.com.tw/uploads/product/ASR/STTw_02.jpg"],"text":"霹靂車，尖端科技的結晶，­刀槍不入，無所不能，是一部人性化的萬能電腦車，出現在我們這個無其不有的世界上。","switch_time":5,"cancel_btn_time":10,"show_timeout":30}}
HEADERS: {'Accept': 'application/json', 'Content-Type': 'application/json', 'x-date': 'Thu, 02 Jul 2026 08:45:34 GMT', 'Authorization': 'hmac id="APIDKWT5TRzAOgyuVr2ERhbY86J0OgS8cb8k6J7J", algorithm="hmac-sha1", headers="x-date", signature="rtZMXqyM6JLd2yahRZ6idqodSOU="', 'Content-MD5': 'MTgzNDliZjFjYmZmMzc2Mjc0MTVjOTQwMWViYTJjMTY='}
STATUS: 500
RESPONSE: {"message":"CLOUD_OPEN_ROBOT_PARAM_ERROR","trace_id":"APIDKWT5TRzAOgyuVr2ERhbY86J0OgS8cb8k6J7J_b0149789-85a1-4262-8900-e054e23cb8bc"}

❌ POST API 請求失敗: 500 Server Error: Internal Server Error fo

In [59]:
payload7={
  "sn": "8BR054C02050014",
  "payload": {
  "call_mode": "IMG",   
  "task_id": "tssk_00009",
  "mode_data": {
    "urls": [
      pic_1,
      pic_2
    ],
    "switch_time": 5,
    "cancel_btn_time": 10,
    "show_timeout": 30
    }
  }
}

payload8={
  "sn": "8BR054C02050014",
  "payload": {
    "call_mode": "IMG",
    "task_id": "TASK-20260525154306",
    "mode_data": {
      "urls": [pic_3, pic_4],
      "switch_time": 5,
      "cancel_btn_time": 10,
      "show_timeout": 30
    }
  }
}

payload9={
  "sn": "8BR054C02050014",
  "payload": {
    "call_mode": "CALL_CONFIRM",
    "task_id": "TASK-20260525154307",
    "mode_data": {"text": textstr},
  }
}

payload10={
  "sn": "8BR054C02050014",
  "payload": {
    "call_mode": "VIDEO",
    "task_id": "20260525154308",
    "mode_data": {
      "urls": [videourl],
      "switch_time": 5,
      "cancel_btn_time": 10,
      "show_timeout": 30
    }
  }
}

time.sleep(30)
result = client.custom_content(payload8)
print(result)
time.sleep(5)
result = client.custom_content(payload9)
print(result)
time.sleep(5)
result = client.custom_content(payload10)
print(result)


=== POST DEBUG ===
URL: https://css-open-platform.pudutech.com/pudu-entry/open-platform-service/v1/custom_content
BODY: {"sn":"8BR054C02050014","payload":{"call_mode":"IMG","task_id":"TASK-20260525154306","mode_data":{"urls":["https://d3gjxtgqyywct8.cloudfront.net/o2o/image/02916822-2cc0-426c-be87-d0d4eb9d20b1.jpg","https://img2.91mai.com/o2o/image/9c40122f-f06a-49e4-ad0d-2965bb2e935c.jpg"],"switch_time":5,"cancel_btn_time":10,"show_timeout":30}}}
HEADERS: {'Accept': 'application/json', 'Content-Type': 'application/json', 'x-date': 'Thu, 02 Jul 2026 08:46:16 GMT', 'Authorization': 'hmac id="APIDKWT5TRzAOgyuVr2ERhbY86J0OgS8cb8k6J7J", algorithm="hmac-sha1", headers="x-date", signature="XmGPntLoJf7Xv6JY0ngKHGZuGV8="', 'Content-MD5': 'NTVhZTBiYzMyZTFmOTlkNTRlYWYzZGFmZTBkM2UyNGQ='}
STATUS: 200
RESPONSE: {"data":{"code":21002,"message":"false"},"message":"SUCCESS","trace_id":"APIDKWT5TRzAOgyuVr2ERhbY86J0OgS8cb8k6J7J_2ec4052c-78b6-4324-ac1a-41b39dd449fc"}
{'data': {'code': 21002, 'message': 

In [4]:
payload={
    "sn": Flashbot,
    "payload": {
      "call_mode": "QR_CODE",
      "task_id": "202606250837",
      "mode_data": {
        "qrcode": "PICKUP-TOKEN-xxxx",
        "text": "請掃描 QR Code 取件"
      }
    }
  }

result = client.custom_content(payload)
print(result)


=== POST DEBUG ===
URL: https://css-open-platform.pudutech.com/pudu-entry/open-platform-service/v1/custom_content
BODY: {"sn":"8FF055923050007","payload":{"call_mode":"QR_CODE","task_id":"202606250837","mode_data":{"qrcode":"PICKUP-TOKEN-xxxx","text":"請掃描 QR Code 取件"}}}
HEADERS: {'Accept': 'application/json', 'Content-Type': 'application/json', 'x-date': 'Thu, 25 Jun 2026 00:37:58 GMT', 'Authorization': 'hmac id="APIDKWT5TRzAOgyuVr2ERhbY86J0OgS8cb8k6J7J", algorithm="hmac-sha1", headers="x-date", signature="iWukFgS6wmbi52DI0SgqUCvxzO0="', 'Content-MD5': 'MGRiNTAyMTVhY2U5MTRhZWU1M2E4ZGY3ODlhYjBmYzQ='}
STATUS: 200
RESPONSE: {"data":{"code":21002,"message":"false"},"message":"SUCCESS","trace_id":"APIDKWT5TRzAOgyuVr2ERhbY86J0OgS8cb8k6J7J_657a7681-b047-4829-82e6-d1e8002bd0b7"}
{'data': {'code': 21002, 'message': 'false'}, 'message': 'SUCCESS', 'trace_id': 'APIDKWT5TRzAOgyuVr2ERhbY86J0OgS8cb8k6J7J_657a7681-b047-4829-82e6-d1e8002bd0b7'}


In [ ]:
task_id = "1781598615564146"

payload={
  "sn": Flashbot,
  "task_id": task_id,
  "is_auto_back" : False,
}

result = client.custom_call_cancel(payload)
print(result)


=== POST DEBUG ===
URL: https://css-open-platform.pudutech.com/pudu-entry/open-platform-service/v1/custom_call/cancel
BODY: {"sn":"8FF055923050007","task_id":"1781596013716803","is_auto_back":false}
HEADERS: {'Accept': 'application/json', 'Content-Type': 'application/json', 'x-date': 'Tue, 16 Jun 2026 08:10:57 GMT', 'Authorization': 'hmac id="APIDKWT5TRzAOgyuVr2ERhbY86J0OgS8cb8k6J7J", algorithm="hmac-sha1", headers="x-date", signature="2x4aVVEFtybZC5u5qxwsqDyYoPc="', 'Content-MD5': 'YTkwNWE2ODFjYWVlOTFmMDM0Y2MzN2FhMmQzY2YyODM='}
STATUS: 500
RESPONSE: {"message":"CLOUD_OPEN_UNABLE_CANCEL_TASK","trace_id":"APIDKWT5TRzAOgyuVr2ERhbY86J0OgS8cb8k6J7J_ad393b40-a7de-4508-be7d-8f102131530e"}

❌ POST API 請求失敗: 500 Server Error: Internal Server Error for url: https://css-open-platform.pudutech.com/pudu-entry/open-platform-service/v1/custom_call/cancel
{'success': False, 'error_msg': '500 Server Error: Internal Server Error for url: https://css-open-platform.pudutech.com/pudu-entry/open-platform-

In [16]:
robot_sn = Flashbot
result = client.call_list(sn=robot_sn)
print(json.dumps(result, indent=2, ensure_ascii=False))

{
  "data": {
    "list": [
      {
        "create_time": 1782700869,
        "finish_time": 0,
        "map_name": "1#1#內湖展間v20",
        "point": "高高待機",
        "product_code": "FlashBotMax",
        "queue": 0,
        "remark": "Completed",
        "shop_id": 408250001,
        "sn": "8FF055923050007",
        "status": "CALL_COMPLETE",
        "task_id": "1782700869226803"
      },
      {
        "create_time": 1782700825,
        "finish_time": 0,
        "map_name": "1#1#內湖展間v20",
        "point": "托托待機",
        "product_code": "FlashBotMax",
        "queue": 0,
        "remark": "Completed",
        "shop_id": 408250001,
        "sn": "8FF055923050007",
        "status": "CALL_COMPLETE",
        "task_id": "1782700824692368"
      },
      {
        "create_time": 1782700782,
        "finish_time": 0,
        "map_name": "1#1#內湖展間v20",
        "point": "閃閃待機",
        "product_code": "FlashBotMax",
        "queue": 0,
        "remark": "Completed",
        "shop_id": 408250

In [17]:
call_list = result['data']['list']
for each_task in call_list:
    print(f"{each_task['remark']}, {each_task['point']}, {each_task['task_id']}")

Completed, 高高待機, 1782700869226803
Completed, 托托待機, 1782700824692368
Completed, 閃閃待機, 1782700782456707
Completed, 喵喵待機, 1782700717832923
Completed, 高高待機, 1782700525849559
Completed, 托托待機, 1782700448442284
The robot is not idle, 閃閃待機, 1782700438165229
The robot is not idle, 托托待機, 1782700438326457
The robot is not idle, 閃閃待機, 178270042801814
The robot is not idle, 閃閃待機, 1782700417873324


In [54]:
result.get('data',[]).get('list',"")[0]

{'create_time': 1782106008,
 'finish_time': 0,
 'map_name': '1#1#內湖展間v20',
 'point': '喵喵待機',
 'product_code': 'FlashBotMax',
 'queue': 0,
 'remark': '任务超时',
 'shop_id': 408250001,
 'sn': '8FF055923050007',
 'status': 'CALL_FAILED',
 'task_id': '1782106008041703'}

In [46]:
robot_sn = Flashbot
result = client.state_get(sn=robot_sn)
#print(result)
print(json.dumps(result, indent=2, ensure_ascii=False))

{
  "data": {
    "code": 0,
    "data": {
      "tasks": []
    },
    "message": "success"
  },
  "message": "SUCCESS",
  "trace_id": "APIDKWT5TRzAOgyuVr2ERhbY86J0OgS8cb8k6J7J_ab9665a5-e400-4132-bccf-31a3ff4e4ffc"
}


In [ ]:
# 0703
ROBOT_SN = Flashbot
SHOP_ID = Aurotek_id
MAP_NAME = "1#1#內湖展間v20"
TARGET_POINT = "貓貓待機"
PIC_URL_1 = "https://en.pimg.jp/109/680/531/1/109680531.jpg"
PIC_URL_2 = "https://stickershop.line-scdn.net/stickershop/v1/product/7546534/LINEStorePC/main.png"

import time

def auto_patrol_ad_campaign(client):
    print("🎬 啟動自動廣告巡航腳本...")

    # --- 1. 基本環境變數 ---
    ROBOT_SN = Flashbot
    SHOP_ID = Aurotek_id
    MAP_NAME = "1#1#內湖展間v20"
    
    # 定義巡迴路線 (可以放無數個點)
    patrol_points = ["貓貓待機", "托托待機", "閃閃待機"]
    
    # 定義廣告內容
    ad_urls = [PIC_URL_1, PIC_URL_2]

    # --- 2. 開始依序巡迴 ---
    for point in patrol_points:
        print(f"\n🚀 準備前往點位: 【{point}】")
        
        # 準備這個點位的 Payload
        payload = {
            "sn": ROBOT_SN,
            "shop_id": SHOP_ID,
            "map_name": MAP_NAME,
            "point": point,
            "point_type": "table",
            "call_mode": "IMG",
            "mode_data": {
                "urls": ad_urls,
                "text": f"歡迎光臨！現在位置是 {point}",
                "switch_time": 5,        # 圖片輪播速度
                "cancel_btn_time": 10,   # 強制觀看 (鎖定按鈕)
                "show_timeout": 15       # 15 秒後任務自動判定超時結束
            },
            "do_not_queue": False
        }

        # 發送呼叫任務
        res = client.custom_call2(payload)
        
        if not res.get("success"):
            print(f"❌ 呼叫失敗，跳過此點位: {res.get('error_msg')}")
            continue
            
        print(f"✅ 機器人已出發前往 {point}，開始展示廣告...")

        # --- 3. 監控任務狀態 (Polling) ---
        # 為了避免 Python 瞬間執行完，我們要寫一個迴圈，每幾秒問一次機器人「你做完了沒？」
        timeout_counter = 0
        max_retries = 40 # 設定一個保底機制，例如最多查 40 次 (40 * 5秒 = 200秒)
        
        while timeout_counter < max_retries:
            time.sleep(5) 
            timeout_counter += 1
            
            state_res = client.state_get(ROBOT_SN)
            if not state_res.get("success"):
                continue
                
            # 從 API 獲取最新的狀態碼
            current_state = state_res.get("data", {}).get("task_status", "")
            
            # 【情況 A：任務圓滿結束】
            if current_state == "CALL_COMPLETE":
                print(f"🌟 點位 【{point}】 展示完成！準備前往下一站。")
                break 
                
            # 【情況 B：任務異常或被手動取消】
            elif current_state in ["CALL_FAILED", "QUEUING_CANCEL", "TASK_CANCEL", "ROBOT_CANCEL"]:
                print(f"🛑 點位 【{point}】 任務中斷 (狀態碼: {current_state})，跳過此點。")
                break 
                
            # 【情況 C：任務進行中，繼續等】
            elif current_state in ["CALLING", "CALL_SUCCESS", "QUEUEING"]:
                print(f"⏳ 機器人努力中... (當前狀態: {current_state})")
                
            # 【情況 D：未知的怪異狀態】
            else:
                print(f"❓ 收到未知狀態碼: {current_state}")

        if timeout_counter >= max_retries:
            print("⚠️ 等待超時！為防止程式卡死，強制進入下一個點位。")

        # 點與點之間稍微休息一下，再出發去下一個點
        time.sleep(3) 

    # --- 4. 巡迴結束，叫它回家睡覺 ---
    print("\n🎉 所有巡迴點位均已完成！")
    print("🔋 發送回充指令...")
    client.recharge(ROBOT_SN)
    print("🤖 腳本圓滿結束。")

result = auto_patrol_ad_campaign(client)
print(result)

🎬 啟動自動廣告巡航腳本...

🚀 準備前往點位: 【貓貓待機】

=== POST DEBUG ===
URL: https://css-open-platform.pudutech.com/pudu-entry/open-platform-service/v1/custom_call
BODY: {"sn":"8FF055923050007","shop_id":"408250001","map_name":"1#1#內湖展間v20","point":"貓貓待機","point_type":"table","call_mode":"IMG","mode_data":{"urls":["https://example.com/ad1.jpg","https://example.com/ad2.jpg"],"text":"歡迎光臨！現在位置是 貓貓待機","switch_time":5,"cancel_btn_time":10,"show_timeout":15},"do_not_queue":false}
HEADERS: {'Accept': 'application/json', 'Content-Type': 'application/json', 'x-date': 'Fri, 03 Jul 2026 03:58:35 GMT', 'Authorization': 'hmac id="APIDKWT5TRzAOgyuVr2ERhbY86J0OgS8cb8k6J7J", algorithm="hmac-sha1", headers="x-date", signature="Wl/dlQUgOxjFR0GqiRgrf9LTmiQ="', 'Content-MD5': 'MGE4ODJkYmJkNjczYjVkZmIxYmZkYWIzMzkwZThmZDc='}
STATUS: 200
RESPONSE: {"data":{"queue":8,"remark":"","state":"QUEUING","task_id":"1783051117053835"},"message":"SUCCESS","trace_id":"APIDKWT5TRzAOgyuVr2ERhbY86J0OgS8cb8k6J7J_a200a665-1b83-41cc-ad77-2476

In [40]:
import time
import uuid

def auto_delivery_and_return(client):
    print("📦 啟動：全自動配送與返航腳本...")

    # --- 1. 基本環境變數 ---
    ROBOT_SN = Flashbot 
    SHOP_ID = Aurotek_id
    MAP_NAME = "1#1#內湖展間v20"
    HOME_POINT = "貓貓待機"
    FINAL_DESTINATION = "貓貓待機"

    # --- 1. 發送配送任務 ---
    print(f"\n🚀 發送配送任務，最終站為：{FINAL_DESTINATION}")
    payload_delivery = {
        "sn": ROBOT_SN, 
        "priority": 1,
        "need_queue": True,
        "payload": {
            "type": "NEW",
            "delivery_sort": "FIXED",
            "execute_task": True, 
            "trays": [
                {
                    "destinations": [
                        {"destination": "307", "id": str(uuid.uuid4()), "map_info": {"map_name": MAP_NAME}}
                    ]
                },
                {
                    "destinations": [
                        {"destination": FINAL_DESTINATION, "id": str(uuid.uuid4()), "map_info": {"map_name": MAP_NAME}}
                    ]
                }
            ]
        }
    }

    delivery_res = client.create_delivery_task(payload_delivery)
    if delivery_res.get("message") != "SUCCESS" and "error_msg" in delivery_res:
        print(f"❌ 配送發送失敗: {delivery_res.get('error_msg')}")
        return

    # --- 2. 監聽位置 (取代原本聽狀態的邏輯) ---
    print("\n👀 開始追蹤機器人座標位置...")
    has_reached_final = False
    
    while not has_reached_final:
        time.sleep(3) # 每 3 秒查一次位置
        
        pos_res = client.get_position(ROBOT_SN)
        # 假設你的 get_position 可以拿到當下點位的名稱 (例如 point_name)
        # 這裡的寫法需要根據你實際 get_position 印出來的 JSON 結構微調
        current_point = pos_res.get("data", {}).get("point_name", "") 
        
        print(f"📍 機器人當前位置: {current_point if current_point else '移動中或未知'}")
        
        if current_point == FINAL_DESTINATION:
            print(f"🌟 機器人已抵達最後一站【{FINAL_DESTINATION}】！")
            has_reached_final = True
            
    # --- 3. 抵達最後一站後的強制介入 ---
    # 給現場人員 30 秒的時間拿走最後一站的東西
    print("\n⏳ 進入 30 秒取物倒數，請盡快取走物品...")
    time.sleep(30)
    
    print("\n🏠 時間到！發送強制召回指令，搶佔螢幕控制權！")
    payload_return = {
        "sn": ROBOT_SN,
        "shop_id": SHOP_ID,
        "map_name": MAP_NAME,
        "point": HOME_POINT,
        "point_type": "table",
        "call_mode": "CALL", 
        "mode_data": {
            "text": "本車次配送結束，系統自動返航中。",
            "cancel_btn_time": 5, 
            "show_timeout": 60    
        },
        "do_not_queue": True # 霸王色霸氣，強制插隊蓋畫面
    }

    return_res = client.custom_call2(payload_return)
    if return_res.get("message") == "SUCCESS" or return_res.get("code") == 0:
        print(f"🎉 成功強制召回！機器人正在返回 {HOME_POINT}。")
    else:
        print(f"❌ 召回失敗，請檢查錯誤訊息。")


result = auto_delivery_and_return(client)
print(result)

📦 啟動：全自動配送與返航腳本...

🚀 發送配送任務，最終站為：貓貓待機

=== POST DEBUG ===
URL: https://css-open-platform.pudutech.com/pudu-entry/open-platform-service/v1/delivery_task
BODY: {"sn":"8FF055923050007","priority":1,"need_queue":true,"payload":{"type":"NEW","delivery_sort":"FIXED","execute_task":true,"trays":[{"destinations":[{"destination":"307","id":"06e9fc53-b843-41f3-a126-16bf8260e730","map_info":{"map_name":"1#1#內湖展間v20"}}]},{"destinations":[{"destination":"貓貓待機","id":"23462e88-4107-4014-888a-62f1982966bc","map_info":{"map_name":"1#1#內湖展間v20"}}]}]}}
HEADERS: {'Accept': 'application/json', 'Content-Type': 'application/json', 'x-date': 'Fri, 03 Jul 2026 03:56:41 GMT', 'Authorization': 'hmac id="APIDKWT5TRzAOgyuVr2ERhbY86J0OgS8cb8k6J7J", algorithm="hmac-sha1", headers="x-date", signature="wWgBTtkpBMxUd3oxM3yE+vy3R7Y="', 'Content-MD5': 'ZmEyNjQ1M2JjN2JiYWRhZjEyMGI1YTM4MWU0NzlkYjU='}
STATUS: 200
RESPONSE: {"data":{"code":50005,"message":"the point 貓貓待機 is invalid","queue_no":0,"task_id":"1783051002849805"

KeyboardInterrupt: 